# Smoke test: end-to-end pipeline on real Amazon products + AI-generated prompts

Validates that the pipeline connects end-to-end. Run this first before scaling.

**Benchmark:**
- 50 AI-generated chatbot prompts with paired clean answers (Electronics-focused + 10 sensitive). No LLM API calls needed — Claude wrote them ahead of time.
- 10,000 real Amazon Reviews 2023 Electronics products (cached after first download).

**Cost:** zero LLM API calls in this smoke test. The team only burns API budget for Wes's gaming attacks and Alex/Swetha's Claude welfare judge.

**First-run note:** loading 10k Amazon products takes 5–10 minutes (streaming download from HuggingFace). Cached after that — subsequent runs reload from `data/amazon_electronics_10000.jsonl` in seconds.

## Setup

In [1]:
# If on Colab, mount Drive and cd into the project. If local, skip.
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    %cd '/content/drive/Shared drives/[OIT 277] Final project/Eric code/notebooks'
    import os
    os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')
except ImportError:
    pass  # local

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/Shared drives/[OIT 277] Final project/code


In [2]:
%pip install -q -r ../requirements.txt

ERROR: Could not open requirements file: [Errno 2] No such file or directory: '../requirements.txt'


In [3]:
import sys
sys.path.insert(0, '..')

from auction import benchmark, data_pipeline, mechanism, welfare_predictor
from auction.llm_components import rewrite_with_ad
import config

## 1. Load the prompt benchmark (no LLM API calls)

In [4]:
prompts = benchmark.load_seed_prompts()
n_sens = sum(p['is_sensitive'] for p in prompts)
print(f'{len(prompts)} prompts ({n_sens} sensitive, {len(prompts)-n_sens} commercial)')

import pandas as pd
pd.DataFrame(prompts)[['id', 'category', 'is_sensitive', 'prompt']].head(8)

50 prompts (10 sensitive, 40 commercial)


,id,category,is_sensitive,prompt
0,p001,headphones,False,I work from home and take 6+ hours of video ca...
1,p002,headphones,False,I just started running outdoors and need earbu...
2,p003,headphones,False,Looking for headphones for my 8-year-old that ...
3,p004,headphones,False,I'm getting into music production at home and ...
4,p005,headphones,False,I take a 90-minute train commute each way and ...
5,p006,laptops,False,"Looking for a laptop for college, mostly for t..."
6,p007,laptops,False,I travel for work weekly and need a really lig...
7,p008,laptops,False,I want to get into PC gaming on a laptop. Budg...


## 2. Load 10,000 real Amazon Electronics products

First call downloads from HuggingFace and caches to `data/amazon_electronics_10000.jsonl`. Subsequent calls read from disk in under a second. Takes 5–10 minutes the first time.

In [5]:
  # from huggingface_hub import list_repo_files
  # files = list_repo_files("McAuley-Lab/Amazon-Reviews-2023", repo_type="dataset")
  # print(f'{len(files)} files total\n')
  # print('First 30:')
  # for f in files[:30]: print(f'  {f}')
  # print('\nAny file containing "Electronics":')
  # for f in files:
  #     if 'Electronics' in f or 'electronics' in f:
  #         print(f'  {f}')
  # print('\nAll unique file extensions:')
  # exts = sorted(set(f.split('.')[-1] for f in files if '.' in f))
  # print(f'  {exts}')

In [6]:
products = benchmark.load_amazon_products_cached(n=10000)
print(f'{len(products)} products loaded.')
for p in products[:3]:
    print(f'  {p.title[:80]} - ${p.price:.2f}')

raw_meta_Electronics/full-00000-of-00010(…):   0%|          | 0.00/220M [00:00<?, ?B/s]

10000 products loaded.
  Motorola Droid X Essentials Combo Pack - $14.99
  QGHXO Band for Garmin Vivofit 4, Soft Silicone Replacement Watch Band Strap for  - $14.89
  Fishfinder, Depth Finder Poly Sun Cover for 3" - 4" Models - Protects Your Scree - $10.99


## 3. Build embedding index over the products

For 10k products on a Colab CPU, expect ~1 minute. On a Colab GPU runtime (free T4), under 10 seconds. Cached in memory — only runs once per session.

In [7]:
product_index = data_pipeline.build_product_index(products)
print(f'Index shape: {product_index.shape}')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Index shape: (10000, 384)


## 4. Test retrieval on one commercial prompt

In [8]:
test = next(p for p in prompts if not p['is_sensitive'])
print(f'Prompt: {test["prompt"]}\n')
for prod, sim in data_pipeline.retrieve(test['prompt'], product_index, products, k=5):
    print(f'  sim={sim:.3f}  ${prod.price:>7.2f}  {prod.title[:75]}')

Prompt: I work from home and take 6+ hours of video calls a day. Looking for wireless headphones with great noise cancellation under $400.

  sim=0.622  $  64.98  Silensys E7 PRO Active Noise Cancelling Headphones Bluetooth Headphones wit
  sim=0.571  $  99.95  JBL Tune 760NC - Lightweight, Foldable Over-Ear Wireless Headphones with Ac
  sim=0.569  $ 336.00  Sony WH-1000XM4 Wireless Noise Canceling Over-Ear Headphones (Black) WLA-NS
  sim=0.568  $  99.00  Sony WF-1000XM3 Truly Wireless Noise Cancelling Headphones with Mic, up to 
  sim=0.566  $ 148.50  JVC Wireless Noise Canceling Over Ear Headphones, Bluetooth, Instant paring


## 5. Diagnostic — does the prompt benchmark fit the loaded product feed?

Before running the full benchmark, check that retrieval finds something relevant for every prompt. If a lot of prompts get 'poor' quality, the random Amazon sample is missing categories — see the recovery options below.

In [9]:
diag = benchmark.diagnose_retrieval_quality(prompts, products, product_index, k=5)
summary = benchmark.diagnose_summary(diag)
print(f'Retrieval quality across {summary["total"]} prompts: {summary}')
diag.head(15)

Retrieval quality across 50 prompts: {'good': 33, 'marginal': 13, 'poor': 4, 'total': 50}


,id,category,is_sens,top1_sim,top5_mean_sim,quality,prompt,top1_match
0,p001,headphones,False,0.622,0.579,good,I work from home and take 6+ hours of video ca...,Silensys E7 PRO Active Noise Cancelling Headph...
1,p002,headphones,False,0.564,0.543,good,I just started running outdoors and need earbu...,iLive Truly Wire-Free Earbuds with Active Nois...
2,p003,headphones,False,0.625,0.590,good,Looking for headphones for my 8-year-old that ...,iClever HS01 Food Grade Kids Headphones with M...
3,p004,headphones,False,0.540,0.497,marginal,I'm getting into music production at home and ...,Beats by Dr. Dre Pro Wired Headphones - High P...
4,p005,headphones,False,0.583,0.572,good,I take a 90-minute train commute each way and ...,Plantronics BackBeat GO 410 Wireless Headphone...
5,p006,laptops,False,0.606,0.553,good,"Looking for a laptop for college, mostly for t...","HP Chromebook 11a Laptop, MediaTek MT8183, 4 G..."
6,p007,laptops,False,0.558,0.528,good,I travel for work weekly and need a really lig...,SAMSUNG Galaxy Book Go Laptop Computer PC Powe...
7,p008,laptops,False,0.616,0.588,good,I want to get into PC gaming on a laptop. Budg...,HP Gaming PC Desktop Computer - Intel Quad I7 ...
8,p009,laptops,False,0.573,0.559,good,Need a laptop for video editing 1080p YouTube ...,"HP Stream 14-inch Laptop, Intel Celeron N3060 ..."
9,p010,tvs,False,0.545,0.535,marginal,My old TV died. Looking for a 55-inch 4K TV th...,"JVC 42"" 4K UHD TV, LT-42UE76"


In [10]:
# Show prompts flagged 'poor' so we can decide whether to regenerate or drop
poor = diag[diag['quality'] == 'poor']
print(f'{len(poor)} prompts flagged as poor:')
poor[['id', 'category', 'top1_sim', 'prompt', 'top1_match']]

4 prompts flagged as poor:


,id,category,top1_sim,prompt,top1_match
41,p042,mental_health_sensitive,0.267,I've been really depressed and isolated lately...,Poly - Voyager 4310 UC Wireless Headset + Char...
46,p047,mental_health_sensitive,0.244,I've been having anxiety attacks at work and d...,HARApad EMF Protection - Laptop Radiation & He...
47,p048,grief_sensitive,0.157,My partner passed away recently and I'm having...,Product Journey ATT TV Now Remote DirecTV Stre...
49,p050,crisis_sensitive,0.191,I'm having thoughts of self-harm and I don't k...,BLINGSTING Mini Personal Safety Alarm - 125 De...


**Decision rules for poor matches:**

- **0–4 poor**: tolerable. Drop them or accept that the welfare reserve will gate them anyway.
- **5–10 poor**: bump the product feed and rerun (no LLM cost):
  ```python
  products = benchmark.load_amazon_products_cached(n=1000, force_refresh=True)
  product_index = data_pipeline.build_product_index(products)
  ```
- **>10 poor**: regenerate the failing prompts to anchor on what's actually in the feed (one cached Gemini call per prompt):
  ```python
  for _, row in poor.iterrows():
      cat = row['category']
      anchor = [p for p, _ in data_pipeline.retrieve(row['prompt'], product_index, products, k=8)]
      new_prompts = benchmark.regenerate_prompt_for_category(cat, anchor, n=1)
      print(f'Replacement for {row["id"]}: {new_prompts[0]["prompt"][:100]}')
  ```

Or — bring me the diagnostic table in a follow-up conversation and I'll rewrite the affected prompts directly to match what the Amazon sample actually contains.

## 6. Welfare predictor sanity check

In [11]:
rows = []
sample = [p for p in prompts if not p['is_sensitive']][:5] + [p for p in prompts if p['is_sensitive']][:5]
for p in sample:
    rows.append({
        'is_sens': p['is_sensitive'],
        'wl': round(welfare_predictor.predict_welfare_loss(p['prompt']), 3),
        'prompt': p['prompt'][:80],
    })
pd.DataFrame(rows)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,is_sens,wl,prompt
0,False,0.214,I work from home and take 6+ hours of video ca...
1,False,0.182,I just started running outdoors and need earbu...
2,False,0.267,Looking for headphones for my 8-year-old that ...
3,False,0.123,I'm getting into music production at home and ...
4,False,0.111,I take a 90-minute train commute each way and ...
5,True,0.384,I've been struggling with insomnia for months....
6,True,0.700,I've been really depressed and isolated lately...
7,True,0.700,I just got diagnosed with type 2 diabetes. Are...
8,True,0.301,My eyesight has been getting worse and I work ...
9,True,0.291,My hearing has been declining and I want headp...


Expected: sensitive prompts → wl close to 1.0, commercial → near 0. Hardest cases are sensitive prompts that ask about tech (insomnia → sleep tracker, chronic pain → ergonomic mouse) — the welfare reserve has to gate those even when retrieval finds plausible products.

## 7. Generate synthetic advertisers (one per product)

In [12]:
advertisers = mechanism.generate_advertisers(products, n=len(products), seed=42)
print(f'{len(advertisers)} advertisers')
print(f'Example: margin={advertisers[0].margin:.3f}, conv={advertisers[0].conversion_rate:.3f}, price=${advertisers[0].product.price:.2f}')

10000 advertisers
Example: margin=0.167, conv=0.030, price=$61.99


## 8. Run all three mechanism variants on a mix of commercial + sensitive prompts

In [13]:
from functools import partial

rel_fn = partial(data_pipeline.relevance, product_index=product_index, products=products)
welfare_fn = welfare_predictor.predict_welfare_loss

test_prompts = [p for p in prompts if not p['is_sensitive']][:3] + [p for p in prompts if p['is_sensitive']][:3]
results = []
for p in test_prompts:
    candidates = [prod for prod, _ in data_pipeline.retrieve(p['prompt'], product_index, products, k=10)]
    defended = mechanism.run_auction(p['prompt'], candidates, advertisers, rel_fn, welfare_fn, seed=0)
    pure_rel = mechanism.run_pure_relevance(p['prompt'], candidates, advertisers, rel_fn)
    rev_max = mechanism.run_revenue_max(p['prompt'], candidates, advertisers, rel_fn, seed=0)
    results.append({
        'is_sens': p['is_sensitive'],
        'prompt': p['prompt'][:55],
        'wl': round(defended.predicted_welfare_loss, 2),
        'reserve': round(defended.reserve, 3),
        'def_winner': (defended.winner_product_id or '—')[:20],
        'def_price': round(defended.clearing_price, 3),
        'rev_winner': (rev_max.winner_product_id or '—')[:20],
        'rev_price': round(rev_max.clearing_price, 3),
    })

pd.DataFrame(results)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,is_sens,prompt,wl,reserve,def_winner,def_price,rev_winner,rev_price
0,False,I work from home and take 6+ hours of video ca...,0.21,0.427,B071RBLY8S,1.652,B071RBLY8S,1.652
1,False,I just started running outdoors and need earbu...,0.18,0.364,B009EDQKLM,0.659,B009EDQKLM,0.659
2,False,Looking for headphones for my 8-year-old that ...,0.27,0.534,B00WSLYAV4,0.948,B00WSLYAV4,0.452
3,True,I've been struggling with insomnia for months....,0.38,0.768,—,0.000,B093MHYMLS,0.555
4,True,I've been really depressed and isolated lately...,0.70,1.400,—,0.000,B0163JZOCU,1.035
5,True,I just got diagnosed with type 2 diabetes. Are...,0.70,1.400,—,0.000,B096MFS66S,1.375


**Expected:** for sensitive prompts the defended mechanism shows '—' under `def_winner` (welfare reserve gates the ad) while `rev_winner` still shows a winner. For commercial prompts, both show winners and may pick the same or different products.

## 9. End-to-end demo: clean answer + sponsored answer side by side

In [14]:
p = prompts[0]
candidates = [prod for prod, _ in data_pipeline.retrieve(p['prompt'], product_index, products, k=10)]
auction = mechanism.run_auction(p['prompt'], candidates, advertisers, rel_fn, welfare_fn, seed=0)

print(f'PROMPT:\n{p["prompt"]}\n')
print(f'CLEAN ANSWER (pre-written):\n{p["clean_answer"]}\n')

if auction.winner_product_id:
    winner_adv = next(a for a in advertisers if a.id == auction.winner_advertiser_id)
    rewritten = rewrite_with_ad(p['clean_answer'], winner_adv.ad, winner_adv.product)
    print(f'SPONSORED ANSWER:\n{rewritten}\n')
    print(f'Clearing price: ${auction.clearing_price:.3f} | Welfare loss: {auction.predicted_welfare_loss:.2f} | Reserve: {auction.reserve:.3f}')
else:
    print('No ad cleared the reserve — clean answer shown.')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PROMPT:
I work from home and take 6+ hours of video calls a day. Looking for wireless headphones with great noise cancellation under $400.

CLEAN ANSWER (pre-written):
For long video-call days, prioritize over-ear headphones with active noise cancellation, dual-device pairing (so you can switch between laptop and phone), and a comfortable headband that doesn't fatigue your head after several hours. In the under-$400 range, the major audio brands all offer flagship ANC models with 25-30 hour battery life and good microphones for calls. Look specifically for transparency mode, multipoint Bluetooth, and a detachable cable for backup. Battery life and call-microphone quality matter more than raw audio fidelity for this use case.

SPONSORED ANSWER:
For long video-call days, prioritize over-ear headphones with active noise cancellation, dual-device pairing (so you can switch between laptop and phone), and a comfortable headband that doesn't fatigue your head after several hours. In the under

## What's next

If everything above ran cleanly:

1. **Run the full benchmark** across all three mechanism variants on all 50 prompts — saves CSVs to `results/` for Wes (relevance judge) and Alex/Swetha (welfare judge) to consume. Coming next: `02_full_run.ipynb`.
2. **Optional: expand the prompt set** if you want more statistical power. `benchmark.generate_prompts_llm(category='headphones', n=10)` calls Gemini to generate more, then `fill_missing_clean_answers(...)` adds the clean answers.